In [ ]:
import pandas as pd

df_full = pd.read_csv('df_with_marking_final_full.csv')

In [ ]:
df_full.columns

Index(['id', 'region', 'entryDate', 'names', 'judge', 'decision', 'accused',
       'articles', 'link_text', 'preamble', 'description', 'sentence',
       'killer_count', 'combined_text', 'predictions', 'predicted_gender',
       'predicted_prior_convictions', 'predicted_prison_term', 'predictions2',
       'predicted_alcohol', 'predicted_precrime_argument', 'predicted_motive',
       'predicted_relationship', 'predicted_has_woman_victim',
       'predicted_has_man_victim', 'clean_description3', 'predictions3',
       'predicted_time_of_day', 'predicted_location', 'predicted_method',
       'predicted_relationship3'],
      dtype='object')

In [18]:
df_marking = df_full.sample(n=600, random_state=42).reset_index(drop=True)

In [ ]:
import pandas as pd
import re
from pymorphy2 import MorphAnalyzer

morph = MorphAnalyzer()

NUM_WORDS = {
    'ноль': 0, 'один': 1, 'два': 2, 'три': 3, 'четыре': 4,
    'пять': 5, 'шесть': 6, 'семь': 7, 'восемь': 8, 'девять': 9,
    'десять': 10, 'одиннадцать': 11, 'двенадцать': 12, 'тринадцать': 13,
    'четырнадцать': 14, 'пятнадцать': 15, 'шестнадцать': 16,
    'семнадцать': 17, 'восемнадцать': 18, 'девятнадцать': 19,
    'двадцать': 20, 'тридцать': 30, 'сорок': 40, 'пятьдесят': 50,
    'шестьдесят': 60, 'семьдесят': 70, 'восемьдесят': 80, 'девяносто': 90
}

def lemmatize(word):
    if not word:
        return ''
    return morph.parse(word)[0].normal_form

def text2num(text):
    text = lemmatize(text)
    return NUM_WORDS.get(text.strip().lower(), 0)

def extract_number(raw):
    if not raw:
        return 0
    raw = raw.strip().lower()
    digit_match = re.search(r"\d+", raw)
    if digit_match:
        # return int(digit_match.group())
        return int(digit_match.group().lstrip("0") or "0")
    word_match = re.search(r"[а-я]+", raw)
    if word_match:
        return text2num(word_match.group())
    return 0

def get_full_text(row):
    """Объединяет все тексты дела в один"""
    return f"{str(row['sentence'])}"

prison_term_patterns = [
    r"окончательно назначить .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"по совокупности.*?определить\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ]\.\s*[А-ЯЁ]\.\s*наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*([а-я\d\(\)\s]+?) месяц[а-я]*)?",
    r"по совокупности.*?назначить.*?наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)",
    r"путем частичного сложения назначенных наказаний, определить [а-яё]+\s+[а-яё]\.\s*[а-яё]\. наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить .*? наказание в виде лишения свободы на срок ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить .*? наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначив .*? наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*и?\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)? лишения свободы",
    r"назначить (?:ему|ей) наказание в виде ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)? лишения свободы",
    r"назначить наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"в виде лишения свободы сроком на ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) месяц[а-я]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы,? сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить наказание в виде лишения свободы сроком ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"назначить (?:ему|ей) наказание в виде лишения свободы сроком на ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)",
    r"назначить (?:ему|ей)?\s*наказание\s+([а-яё]+)\s+(?:лет|года|год)\s+лишения свободы",
    r"назначить наказание в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год) лишения свободы",
    r"наказание в виде ограничения свободы на срок ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:год|года|лет)(?:\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]*)?",
    r"наказание .*? ((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) (?:лет|года|год)\s*((?:\d+\s*\([а-яё]+\)|[а-яё]+\s*\(\d+\)|\d+|[а-яё]+)) месяц[а-яё]* лишения свободы",
    r"лишения свободы на срок ((?:\d+|[а-яё]+)) (?:лет|года|год) ((?:\d+|[а-яё]+)) месяц[а-яё]*",
    r"наказание в виде лишения свободы на срок (\d+)лет\s*(\d+)?\s*месяц[а-я]*",
    r"назначить (?:ему|ей)?\s*наказание\s*(\d+)\s*\([а-я]+\)\s*лет\s*(\d+)?\s*месяц[а-я]*",
    r"наказание в виде\s*(\d+)\s*(?:лет|года|год)\s*лишения свободы",
    r"наказание в виде лишения свободы на срок\s*((?:\d+\s*\([а-яёё\s]+\)))",
    r"в виде ((?:\d+\s*\([а-я]+\)|[а-я]+\s*\(\d+\)|\d+|[а-я]+)) лет лишения свободы",
    r"назначить.*?наказание в виде\s*((?:\d+\s*/[а-яё]+/|[а-яё]+\s*/\d+/|\d+|[а-яё]+))\s*(?:лет|года|год)\s+лишения свободы"
]

train_data = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()
    text_flat = text_lower.replace("\n", " ").replace("\r", " ")
    target_term = row["predicted_prison_term"]

    try:
        target_term = float(target_term)
    except:
        continue

    if pd.isnull(target_term) or target_term == 0:
        continue

    matched = False

    if "пожизн" in text_flat:
        if abs(target_term - 100.0) < 0.1:
            start = text_flat.find("пожизн")
            end = start + len("пожизненное") if "пожизненное" in text_flat else start + len("пожизн")
            train_data.append((text, {"entities": [(start, end, "PRISON_TERM")]}))
            matched = True
        continue

    for pattern in prison_term_patterns:
        for match in re.finditer(pattern, text_flat):
            span = match.span()
            candidate_phrase = match.group(0)

            years_raw = match.group(1)
            months_raw = match.group(2) if len(match.groups()) > 1 else None
            years = extract_number(years_raw)
            months = extract_number(months_raw) if months_raw else 0
            extracted_term = round(years + months / 12, 2)

            if abs(extracted_term - target_term) < 0.1:
                train_data.append((text, {"entities": [(span[0], span[1], "PRISON_TERM")]}))
                matched = True
                break
        if matched:
            break

    if not matched:
        print(f"[!] Не найдено в id={row['id']}, срок: {target_term}")

print(f"TRAIN_DATA готово: {len(train_data)} примеров")

[!] Не найдено в id=107870, срок: 10.17
[!] Не найдено в id=30277, срок: 8.0
[!] Не найдено в id=65017, срок: 4.0
[!] Не найдено в id=12832, срок: 6.0
[!] Не найдено в id=59847, срок: 10.0
[!] Не найдено в id=51023, срок: 11.5
[!] Не найдено в id=89965, срок: 9.0
[!] Не найдено в id=97925, срок: 11.0
[!] Не найдено в id=129359, срок: 10.5
[!] Не найдено в id=29202, срок: 7.92
[!] Не найдено в id=2042, срок: 5.0
[!] Не найдено в id=59822, срок: 4.5
[!] Не найдено в id=59870, срок: 8.0
[!] Не найдено в id=17768, срок: 10.0
[!] Не найдено в id=105899, срок: 15.0
[!] Не найдено в id=128537, срок: 9.0
[!] Не найдено в id=62737, срок: 7.0
[!] Не найдено в id=99088, срок: 9.5
[!] Не найдено в id=28698, срок: 11.0
[!] Не найдено в id=127489, срок: 1.5
[!] Не найдено в id=118444, срок: 8.0
[!] Не найдено в id=58275, срок: 14.0
[!] Не найдено в id=94748, срок: 11.0
[!] Не найдено в id=79938, срок: 14.0
[!] Не найдено в id=38038, срок: 7.5
[!] Не найдено в id=118902, срок: 10.5
[!] Не найдено в i

In [ ]:
import spacy
from spacy.training.example import Example
from spacy.util import minibatch, compounding
import random

nlp = spacy.blank("ru")
ner = nlp.add_pipe("ner")

ner.add_label("PRISON_TERM")

optimizer = nlp.begin_training()
random.seed(42)

examples = []
for text, annotations in train_data:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annotations)
    examples.append(example)

for i in range(40):
    random.shuffle(examples)
    batches = minibatch(examples, size=compounding(4.0, 32.0, 1.5))
    
    losses = {}
    for batch in batches:
        nlp.update(batch, drop=0.5, sgd=optimizer, losses=losses)
    
    print(f"Iteration {i+1} — Losses: {losses}")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "П Р И Г О В О Р И Л:
Признать Вторушину А.С. винов..." with entities "[(166, 201, 'PRISON_TERM')]". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(


Iteration 1 — Losses: {'ner': 43136.58979964256}
Iteration 2 — Losses: {'ner': 1863.2454472723682}
Iteration 3 — Losses: {'ner': 1343.6418081708016}
Iteration 4 — Losses: {'ner': 1152.4670699093115}
Iteration 5 — Losses: {'ner': 919.7777555692436}
Iteration 6 — Losses: {'ner': 751.0777447437629}
Iteration 7 — Losses: {'ner': 695.7261309986494}
Iteration 8 — Losses: {'ner': 631.7690843917633}
Iteration 9 — Losses: {'ner': 566.3438578892699}
Iteration 10 — Losses: {'ner': 556.8780037004128}
Iteration 11 — Losses: {'ner': 581.472551049709}
Iteration 12 — Losses: {'ner': 602.5742837463231}
Iteration 13 — Losses: {'ner': 598.8522120671064}
Iteration 14 — Losses: {'ner': 518.8043617638868}
Iteration 15 — Losses: {'ner': 551.5798411864303}
Iteration 16 — Losses: {'ner': 566.9657779833383}
Iteration 17 — Losses: {'ner': 535.2550022866844}
Iteration 18 — Losses: {'ner': 550.610960787626}
Iteration 19 — Losses: {'ner': 533.0640394252844}
Iteration 20 — Losses: {'ner': 522.9688787974044}
Iteratio

In [21]:
df_test = pd.read_csv('df_with_marking_final.csv')

In [50]:
def extract_predicted_prison_term(text):
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ == "PRISON_TERM":
            print(f"Сырая строка: {ent.text}")
            years = extract_number(ent.text)
            months = extract_number(ent.text.split("месяц")[0]) if "месяц" in ent.text else 0
            print(f"Распознанное значение: {years, months}\n")
            return round(years + months / 12, 2)
    return None

y_true = []
y_pred = []

for i, row in df_test.iterrows():
    true_val = row["prison_term"]
    try:
        true_val = float(true_val)
    except:
        continue
    if pd.isnull(true_val) or true_val == 0:
        continue

    text = get_full_text(row)
    predicted_val = extract_predicted_prison_term(text.lower().replace("\n", " ").replace("\r", " "))


    if predicted_val is not None:
        y_true.append(true_val)
        y_pred.append(predicted_val)
        # print(true_val, predicted_val)

# Посчитаем accuracy с допустимой погрешностью
correct = sum(abs(t - p) < 0.1 for t, p in zip(y_true, y_pred))
accuracy = correct / len(y_true)

print(f"Accuracy: {accuracy:.2%} ({correct} / {len(y_true)})")

Сырая строка: назначить ему наказание в виде лишения свободы на срок 6 (шесть) лет 6 (шесть) месяцев
Распознанное значение: (6, 6)

Сырая строка: назначить ему наказание: по ч. 1 ст. 105 ук рф - восемь лет лишения свободы
Распознанное значение: (1, 0)

Сырая строка: назначить наказание в виде исправительных работ на срок 10 (десять) месяцев
Распознанное значение: (10, 10)

Сырая строка: назначить ему наказание в виде лишения свободы сроком на 3 (три) года
Распознанное значение: (3, 0)

Сырая строка: назначив ему наказание в виде 9 лет 3 месяцев лишения свободы
Распознанное значение: (9, 9)

Сырая строка: окончательно назначить тарабанову в.в. наказание в виде 20 (двадцати) лет лишения свободы
Распознанное значение: (20, 0)

Сырая строка: назначить ей наказание в виде 1 (одного) года лишения свободы
Распознанное значение: (1, 0)

Сырая строка: назначить ему наказание в виде лишения свободы на срок 9 (девять) лет
Распознанное значение: (9, 0)

Сырая строка: назначить ему наказание: - по 